### SCM drug exposure — post-run quality checks

Run top-to-bottom on **SQL warehouse**. Catalog `_exponent` unless your workspace differs.

**Targets:** `omop_silver.drug_exposure` (merge target) and `omop_scm.drug_exposure` (SCM CDM slice).


#### 1. Mapping coverage — standard and source concepts


In [ ]:
%sql
SELECT
  COUNT(*) AS n,
  SUM(CASE WHEN drug_concept_id <> 0 THEN 1 ELSE 0 END) AS mapped_standard,
  ROUND(100.0 * SUM(CASE WHEN drug_concept_id <> 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mapped_standard,
  SUM(CASE WHEN drug_source_concept_id <> 0 THEN 1 ELSE 0 END) AS mapped_source,
  ROUND(100.0 * SUM(CASE WHEN drug_source_concept_id <> 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mapped_source
FROM _exponent.omop_silver.drug_exposure
WHERE source_system = 'allscripts_scm';

In [ ]:
%sql
SELECT
  COUNT(*) AS n,
  SUM(CASE WHEN drug_concept_id <> 0 THEN 1 ELSE 0 END) AS mapped_standard,
  ROUND(100.0 * SUM(CASE WHEN drug_concept_id <> 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mapped_standard
FROM _exponent.omop_scm.drug_exposure;

#### 2. Grain — total vs distinct merge key (post-dedupe silver rows)


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM _exponent.omop_silver.drug_exposure WHERE source_system = 'allscripts_scm') AS row_count,
  (SELECT COUNT(*) FROM (
    SELECT DISTINCT drug_exposure_source_value, drug_type_concept_id
    FROM _exponent.omop_silver.drug_exposure
    WHERE source_system = 'allscripts_scm'
  ) x) AS distinct_keys

#### 3. Person bridge — hydrated rows should resolve to source_to_person


In [ ]:
%sql
SELECT COUNT(*) AS rows_missing_person_map
FROM _exponent.omop_silver.drug_exposure d
WHERE d.source_system = 'allscripts_scm'
  AND NOT EXISTS (
    SELECT 1 FROM _exponent.omop_mapping.source_to_person stp
    WHERE stp.person_source_value = d.person_source_value
      AND stp.active_flag = TRUE
  );

#### 4. Date coherence


In [ ]:
%sql
SELECT
  SUM(CASE WHEN drug_exposure_end_date < drug_exposure_start_date THEN 1 ELSE 0 END) AS bad_end_before_start
FROM _exponent.omop_silver.drug_exposure
WHERE source_system = 'allscripts_scm';

#### 5. OMOP concept sanity — random sample of mapped drugs


In [ ]:
%sql
SELECT d.drug_exposure_source_value, d.drug_type_concept_id, d.drug_concept_id,
       c.domain_id, c.standard_concept, c.vocabulary_id, c.invalid_reason
FROM _exponent.omop_silver.drug_exposure d
JOIN _exponent.omop.concept c ON c.concept_id = d.drug_concept_id
WHERE d.source_system = 'allscripts_scm'
  AND d.drug_concept_id <> 0
LIMIT 500

#### 6. Downstream era checks

After running `allscripts_scm_drug_era` and `allscripts_scm_dose_era` notebooks:


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM _exponent.omop_scm.drug_exposure WHERE drug_concept_id <> 0) AS eligible_drug_exposure_rows,
  (SELECT COUNT(*) FROM _exponent.omop_scm.drug_era) AS drug_era_rows,
  (SELECT COUNT(*) FROM _exponent.omop_scm.drug_exposure WHERE drug_concept_id <> 0 AND (days_supply IS NULL OR days_supply >= 0)) AS eligible_for_dose_era,
  (SELECT COUNT(*) FROM _exponent.omop_scm.dose_era) AS dose_era_rows;

#### 7. DQD

Re-run your SCM DQD package against the CDM build; update `docs/scm_failures.csv` when official thresholds are met.
